In [1]:
import re
import random
from PIL import Image
from IPython.display import display, HTML
import pandas as pd
import numpy as np 
import requests
import datetime
from bs4 import BeautifulSoup 
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

In [2]:
BASE_COLS = ["season", "team", "period", "field", "gf", "ga", "result", "goal_diff", "match",]
PERIOD_MAP = pd.DataFrame({"period": [1, 2, 3, "ot", "so"]})

In [3]:
results = "https://www.hokej.cz/tipsport-extraliga/zapasy?matchList-view-displayAll=1&matchList-filter-season=2025&matchList-filter-competition=7397"

In [43]:
def aggregate_match(match: pd.DataFrame,) -> list[int]:
    gf = match["gf"].sum()
    ga = match["ga"].sum()
    goal_diff = gf - ga 
    periods = len(match)
    
    if goal_diff > 0:
        result_map = {3: "V", 4: "VP", 5: "VN"}
    elif goal_diff < 0:
        result_map = {3: "P", 4: "PP", 5: "PN"}
    else:
        result_map = {}

    result = result_map.get(periods, "R")

    return pd.DataFrame(
        [[gf, ga, goal_diff, result, "F"]], 
        columns=["gf", "ga", "goal_diff", "result", "period"],
                       )

In [44]:
def finalize_results(
    period: pd.DataFrame, 
    team: str, 
    field: str, 
    season_name: str,
    match: int,
                    ) -> pd.DataFrame: 
    
    team_res = period.copy()
    
    agg_res = aggregate_match(team_res)

    team_res["goal_diff"] = team_res["gf"] - team_res["ga"]
    team_res["result"] = pd.cut(
        team_res["gf"] - team_res["ga"],
        bins=[-float("inf"), -1, 0, float("inf")],
        labels=["P", "R", "V"]
                                )

    team_res = pd.concat([team_res, agg_res], ignore_index=True)   
    
    team_res["team"] = team
    team_res["field"] = field
    team_res["season"] = season_name
    team_res["match"] = match
    
    
    return team_res[BASE_COLS]

In [45]:
def period_extraction(period_raw: str) -> pd.DataFrame:
    
    scores = re.findall(r"(\d+):(\d+)", period_raw)
    results = [[int(a), int(b)] for a, b in scores]
    results_df = pd.DataFrame(np.array(results), columns=["gf", "ga"])

    return pd.merge(results_df, PERIOD_MAP, left_index=True, right_index=True)

In [49]:
def extract_round_results(table_round: str, season_name: str) -> [pd.DataFrame, str]: 
    
    round_results = []
    
    for tr in table_round.find_all("tr", class_="js-preview__link"):
        
        team_h, team_a = [span.text.strip() for span in tr.find_all("span", class_="preview__name--short")]
        match_nr = tr.get("data-href").split("/")[-1] if tr.get("data-href") else None
      
        try:
            period_td = tr.find("td", class_="preview__period").get_text(" ", strip=True)
            period_results = period_extraction(period_td)
            round_results.append(finalize_results(period_results, team_h, "D", season_name, match_nr))
            round_results.append(
                finalize_results(period_results.rename(
                    columns={"ga": "gf", "gf": "ga", "period": "period"}), team_a, "V", season_name, match_nr
                                                       )
                                )
            
        except: 
            print(f" {team_h} vs.{team_a}", end=" ")

    if len(round_results) > 0:
        return pd.concat(round_results, ignore_index=True)

In [50]:
def extract_table(url_link) -> pd.DataFrame: 
    
    season = int(re.search(r'season=(\d+)', url_link).group(1))
    season_name = str(season) + "/" + str(season+1)
    r = requests.get(url_link)
    bs = BeautifulSoup(r.content, 'html.parser')
    table = bs.find_all("table", class_="preview")
    
    complete_rounds = []

    print("Not played matches:", end=" ")
    for table_round in table:
        complete_rounds.append(extract_round_results(table_round, season_name))
    print("\n>>> data fetched <<<")
    
    return pd.concat(complete_rounds, ignore_index=True)       

In [51]:
%%time
tables_df = extract_table(results)

Not played matches:  KVA vs.LIB  PCE vs.KOM  MBL vs.SPA  PLZ vs.SPA  SPA vs.MHK  TRI vs.KVA  MBL vs.MHK  LIB vs.CEB  MBL vs.KLA  VIT vs.KVA  LIB vs.PLZ  MHK vs.PCE  OLO vs.TRI  KOM vs.LIT  SPA vs.CEB  PCE vs.LIB  TRI vs.SPA  KLA vs.VIT  KVA vs.LIT  CEB vs.MHK  KOM vs.MBL  PLZ vs.OLO  SPA vs.KOM  CEB vs.PLZ  MBL vs.PCE  VIT vs.LIT  LIB vs.TRI  MHK vs.KLA  OLO vs.KVA  VIT vs.KOM  KLA vs.OLO  LIB vs.LIT  KVA vs.SPA  PLZ vs.MHK  MBL vs.TRI  PCE vs.CEB  SPA vs.LIT  PLZ vs.TRI  MBL vs.VIT  CEB vs.KLA  OLO vs.PCE  MHK vs.KVA  LIB vs.KOM  TRI vs.CEB  LIT vs.OLO  KVA vs.MBL  KLA vs.LIB  PCE vs.PLZ  KOM vs.MHK  SPA vs.OLO  PLZ vs.KOM  CEB vs.KVA  MBL vs.LIT  LIB vs.VIT  PCE vs.KLA  MHK vs.TRI  TRI vs.PCE  LIT vs.MHK  KLA vs.PLZ  KVA vs.LIB  SPA vs.MBL  KOM vs.CEB  MBL vs.OLO  PLZ vs.KVA  CEB vs.LIT  KLA vs.TRI  MHK vs.VIT  LIB vs.SPA  PCE vs.KOM  LIT vs.PCE  MBL vs.LIB  VIT vs.CEB  OLO vs.MHK  KVA vs.KLA  KOM vs.TRI  SPA vs.PLZ  PCE vs.KVA  TRI vs.VIT  KLA vs.KOM  LIB vs.OLO  MHK vs.SPA  PLZ vs.

In [54]:
tables_df.loc[lambda x: x["match"]=="2921771"]

,season,team,period,field,gf,ga,result,goal_diff,match
56,2025/2026,VIT,1,D,0,0,R,0,2921771
57,2025/2026,VIT,2,D,0,0,R,0,2921771
58,2025/2026,VIT,3,D,0,0,R,0,2921771
59,2025/2026,VIT,ot,D,0,0,R,0,2921771
60,2025/2026,VIT,so,D,1,0,V,1,2921771
61,2025/2026,VIT,F,D,1,0,VN,1,2921771
62,2025/2026,KLA,1,V,0,0,R,0,2921771
63,2025/2026,KLA,2,V,0,0,R,0,2921771
64,2025/2026,KLA,3,V,0,0,R,0,2921771
65,2025/2026,KLA,ot,V,0,0,R,0,2921771


In [53]:
tables_df.head()

,season,team,period,field,gf,ga,result,goal_diff,match
0,2025/2026,CEB,1,D,1,1,R,0,2921768
1,2025/2026,CEB,2,D,0,1,P,-1,2921768
2,2025/2026,CEB,3,D,1,1,R,0,2921768
3,2025/2026,CEB,F,D,2,3,P,-1,2921768
4,2025/2026,SPA,1,V,1,1,R,0,2921768
